# Chapter 5: Probability Foundations

<a href="../lite/lab/index.html?path=ch05_probability_foundations.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Probability is the language of uncertainty. In robotics, **nothing is certain**, sensors are noisy,
motors are imprecise, and the world is unpredictable. Instead of pretending we know exactly
where the robot is, we maintain a **probability distribution** over possible states.

This chapter introduces the probability tools we will use throughout the book. Every concept
is motivated by a robotics scenario: Where is the robot? What did the sensor measure?
How confident are we?

```{admonition} What you will build
:class: tip

- Update a robot's position belief using Bayes rule with a LiDAR measurement
- Combine multiple noisy sensors using inverse variance weighting (the idea behind Kalman filtering)
- Show how sequential Bayes updates make the belief sharper with each measurement
- Test conditional independence of sensor readings given the robot state

**Real world application:** Bayes rule is the engine of all probabilistic robotics. After this chapter, you can implement the core update step used in Kalman filters, particle filters, and SLAM.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **scipy.stats** | Probability distributions (`norm`, `multivariate_normal`), statistical tests |
| **PyMC / Stan** | Probabilistic programming for complex Bayesian inference beyond simple conjugate models |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 5.1 Random Variables and Distributions

A **random variable** $X$ represents a quantity whose value is uncertain.
In robotics, examples include:

| Random variable | What it represents | Type |
|---|---|---|
| $X$ | Robot's x-position (meters) | Continuous |
| $Z$ | LiDAR range reading (meters) | Continuous |
| $D$ | Is there a door at this location? | Discrete (yes/no) |
| $C$ | Which landmark am I seeing? | Discrete (1, 2, …, N) |

A **probability distribution** assigns a probability (or density) to each possible value.

For **discrete** variables: $p(X = x) \geq 0$ and $\sum_x p(X = x) = 1$

For **continuous** variables: $p(x) \geq 0$ and $\int p(x) \, dx = 1$

The most important continuous distribution in robotics is the **Gaussian** (normal):

$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left\{-\frac{(x - \mu)^2}{2\sigma^2}\right\}$$

### Example: Where is the robot?

A robot drives down a corridor. Its position is uncertain, we model it as a random variable
with a Gaussian distribution centered on our best estimate.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
mu = 5.0           # believed position (meters)      (try 3, 7)
sigma = 1.2        # uncertainty (std dev, meters)    (try 0.5, 2.0, 3.0)
n_samples = 500    # number of random samples to draw
# ──────────────────────────────────────────────────────────────────────────────

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
pdf = stats.norm.pdf(x, mu, sigma)

np.random.seed(42)
samples = np.random.normal(mu, sigma, n_samples)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PDF
ax = axes[0]
ax.fill_between(x, pdf, alpha=0.3, color='steelblue')
ax.plot(x, pdf, 'steelblue', lw=2, label=f'$\\mathcal{{N}}({mu}, {sigma}^2)$')
ax.axvline(mu, color='k', ls='--', lw=1, label=f'mean $\\mu = {mu}$')
ax.axvspan(mu - sigma, mu + sigma, alpha=0.1, color='orange', label=f'±1σ (68.3%)')
ax.set_xlabel("Position x (meters)"); ax.set_ylabel("Probability density p(x)")
ax.set_title("Probability density function (PDF)"); ax.legend()

# Right: samples as a histogram
ax = axes[1]
ax.hist(samples, bins=40, density=True, alpha=0.5, color='tomato', label=f'{n_samples} samples')
ax.plot(x, pdf, 'steelblue', lw=2, label='true PDF')
ax.set_xlabel("Position x (meters)"); ax.set_ylabel("Density")
ax.set_title("Samples from the distribution"); ax.legend()

plt.tight_layout()
plt.show()

# Verify the area under the curve
area = np.trapz(pdf, x)
print(f"Area under PDF: {area:.4f}  (should be ≈ 1.0)")
print(f"Prob(robot within ±1σ of mean): {stats.norm.cdf(mu+sigma, mu, sigma) - stats.norm.cdf(mu-sigma, mu, sigma):.3f}")

### Example: Discrete distribution, door detection

A robot has a simple door detector. At each position, the detector returns one of three
outcomes: **door**, **wall**, or **open space**. The probability of each outcome depends on
what is actually there.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
# True state: what is actually at this location?
# Probabilities of sensor reading given truth
p_detect_door = [0.8, 0.15, 0.05]    # [door, wall, open] when TRUE=door
p_detect_wall = [0.1, 0.8, 0.1]      # when TRUE=wall
p_detect_open = [0.05, 0.1, 0.85]    # when TRUE=open
true_state = "door"                    # try "door", "wall", "open"
# ──────────────────────────────────────────────────────────────────────────────

probs = {"door": p_detect_door, "wall": p_detect_wall, "open": p_detect_open}
p = probs[true_state]
labels = ["door", "wall", "open"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, p, color=['steelblue', 'tomato', 'orange'], alpha=0.8, edgecolor='k')
for bar, prob in zip(bars, p):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{prob:.2f}', ha='center', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1); ax.set_ylabel("Probability")
ax.set_title(f"Sensor reading distribution when true state = '{true_state}'", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Sum of probabilities: {sum(p):.2f}  (must equal 1.0)")

**Key observations:**
- A **PDF** gives the density, not the probability, for continuous variables, probability is the area under the curve.
- The Gaussian is fully described by just two numbers: mean $\mu$ and variance $\sigma^2$.
- Discrete distributions are simply a table of probabilities that sum to 1.

## 5.2 Joint, Marginal, and Conditional Probability

When we have **two or more** random variables, we need to describe how they relate.

**Joint probability** $p(x, z)$: the probability that $X = x$ AND $Z = z$ simultaneously.

**Marginal probability**: obtained by summing (or integrating) out the other variable:

$$p(x) = \int p(x, z) \, dz \qquad p(z) = \int p(x, z) \, dx$$

**Conditional probability**: the probability of $X$ given that we know $Z = z$:

$$p(x \mid z) = \frac{p(x, z)}{p(z)}$$

### Example: Robot position and LiDAR reading

A robot's position $X$ and its LiDAR range reading $Z$ are correlated, the range depends
on how far the robot is from the wall. We can visualize their joint distribution.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
wall_position = 10.0        # wall is at x = 10 meters
sigma_x = 2.0               # uncertainty in robot position (m)
sigma_z = 0.5               # LiDAR noise std dev (m)
mu_x = 5.0                  # believed robot position (m)
# ──────────────────────────────────────────────────────────────────────────────

# Joint distribution: p(x, z)
# x ~ N(mu_x, sigma_x^2)
# z | x ~ N(wall_position - x, sigma_z^2)   (range = distance to wall + noise)

x_grid = np.linspace(0, 10, 200)
z_grid = np.linspace(0, 12, 200)
X, Z = np.meshgrid(x_grid, z_grid)

# p(x) * p(z|x)
p_x = stats.norm.pdf(X, mu_x, sigma_x)
expected_range = wall_position - X
p_z_given_x = stats.norm.pdf(Z, expected_range, sigma_z)
p_joint = p_x * p_z_given_x

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Joint distribution
ax = axes[0]
ax.contourf(X, Z, p_joint, levels=20, cmap='Blues')
ax.set_xlabel("Robot position x (m)"); ax.set_ylabel("LiDAR range z (m)")
ax.set_title("Joint $p(x, z)$", fontsize=13)
ax.plot(x_grid, wall_position - x_grid, 'r--', lw=1.5, label='$z = 10 - x$ (ideal)')
ax.legend()

# Marginal p(x)
ax = axes[1]
p_x_marginal = np.trapz(p_joint, z_grid, axis=0)
p_x_marginal /= np.trapz(p_x_marginal, x_grid)  # normalize
ax.fill_between(x_grid, p_x_marginal, alpha=0.3, color='steelblue')
ax.plot(x_grid, p_x_marginal, 'steelblue', lw=2)
ax.set_xlabel("Robot position x (m)"); ax.set_ylabel("Density")
ax.set_title("Marginal $p(x) = \\int p(x,z) \\, dz$", fontsize=13)

# Conditional p(x | z = 5)
ax = axes[2]
z_observed = 5.0
z_idx = np.argmin(np.abs(z_grid - z_observed))
p_x_given_z = p_joint[z_idx, :]
p_x_given_z = p_x_given_z / np.trapz(p_x_given_z, x_grid)  # normalize
ax.fill_between(x_grid, p_x_given_z, alpha=0.3, color='tomato')
ax.plot(x_grid, p_x_given_z, 'tomato', lw=2, label=f'$p(x \\mid z = {z_observed})$')
ax.plot(x_grid, p_x_marginal, 'steelblue', lw=1.5, ls='--', alpha=0.5, label='$p(x)$ prior')
ax.set_xlabel("Robot position x (m)"); ax.set_ylabel("Density")
ax.set_title(f"Conditional $p(x \\mid z = {z_observed})$", fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

**Key observations:**
- The **joint distribution** captures the full relationship between position and sensor reading.
- The **marginal** $p(x)$ is our belief about position before we look at the sensor, the **prior**.
- The **conditional** $p(x \mid z)$ is our belief about position *after* seeing the sensor reading, the **posterior**. It is typically narrower (more certain) than the prior.
- This is the fundamental pattern in robotics: **observe, then update your belief**.

## 5.3 Bayes Rule

**Bayes rule** is the mathematical engine of probabilistic robotics. It lets us compute
the posterior $p(x \mid z)$ from the likelihood $p(z \mid x)$ and the prior $p(x)$:

$$p(x \mid z) = \frac{p(z \mid x) \; p(x)}{p(z)} = \eta \; p(z \mid x) \; p(x)$$

where $\eta = p(z)^{-1}$ is a normalizing constant.

**In robotics terms:**

| Term | Meaning | Example |
|------|---------|---------|
| $p(x)$ | **Prior**, what we believed before | "The robot is probably near x = 5" |
| $p(z \mid x)$ | **Likelihood**, sensor model | "If the robot is at x, how likely is this LiDAR reading?" |
| $p(x \mid z)$ | **Posterior**, updated belief | "Given the reading, the robot is probably near x = 5.2" |
| $\eta$ | **Normalizer**, ensures the posterior integrates to 1 | Computed automatically |

### Example: 1D robot localization with Bayes rule

A robot is somewhere in a corridor. It has a prior belief about its position,
then takes a LiDAR measurement. Bayes rule combines them.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
# Prior: robot believes it's near x = 5
prior_mu = 5.0
prior_sigma = 2.0

# Sensor model: LiDAR says wall distance is z = 4.5m
# Wall is at x = 10, so implied position is x = 10 - z = 5.5
z_measured = 4.5
sensor_sigma = 0.8        # LiDAR noise (try 0.3, 1.0, 2.0)
wall_x = 10.0
# ──────────────────────────────────────────────────────────────────────────────

x = np.linspace(0, 10, 500)

# Prior: p(x)
prior = stats.norm.pdf(x, prior_mu, prior_sigma)

# Likelihood: p(z | x) — given position x, what's the probability of measuring z?
# z = wall_x - x + noise, so p(z | x) = N(z; wall_x - x, sensor_sigma^2)
likelihood = stats.norm.pdf(z_measured, wall_x - x, sensor_sigma)

# Unnormalized posterior
posterior_unnorm = likelihood * prior

# Normalize
posterior = posterior_unnorm / np.trapz(posterior_unnorm, x)

# Compute posterior mean and std
post_mean = np.trapz(x * posterior, x)
post_var = np.trapz((x - post_mean)**2 * posterior, x)
post_std = np.sqrt(post_var)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Prior
ax = axes[0]
ax.fill_between(x, prior / np.trapz(prior, x), alpha=0.3, color='steelblue')
ax.plot(x, prior / np.trapz(prior, x), 'steelblue', lw=2)
ax.axvline(prior_mu, color='steelblue', ls=':', lw=1.5)
ax.set_title("Prior $p(x)$", fontsize=13)
ax.set_xlabel("Position (m)"); ax.set_ylabel("Density")

# Likelihood
ax = axes[1]
like_norm = likelihood / np.trapz(likelihood, x)
ax.fill_between(x, like_norm, alpha=0.3, color='orange')
ax.plot(x, like_norm, 'orange', lw=2)
ax.axvline(wall_x - z_measured, color='orange', ls=':', lw=1.5)
ax.set_title(f"Likelihood $p(z={z_measured} \\mid x)$", fontsize=13)
ax.set_xlabel("Position (m)"); ax.set_ylabel("Density")

# Posterior
ax = axes[2]
ax.fill_between(x, posterior, alpha=0.3, color='tomato')
ax.plot(x, posterior, 'tomato', lw=2, label=f'posterior (μ={post_mean:.2f}, σ={post_std:.2f})')
ax.plot(x, prior / np.trapz(prior, x), 'steelblue', lw=1, ls='--', alpha=0.5, label='prior')
ax.plot(x, like_norm, 'orange', lw=1, ls='--', alpha=0.5, label='likelihood')
ax.axvline(post_mean, color='tomato', ls=':', lw=1.5)
ax.set_title("Posterior $p(x \\mid z) \\propto p(z \\mid x) \\, p(x)$", fontsize=13)
ax.set_xlabel("Position (m)"); ax.set_ylabel("Density")
ax.legend(fontsize=10)

plt.suptitle("Bayes Rule: Prior × Likelihood → Posterior", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Prior:     μ = {prior_mu:.2f},  σ = {prior_sigma:.2f}")
print(f"Sensor:    z = {z_measured:.2f} → implied x = {wall_x - z_measured:.2f},  σ_z = {sensor_sigma:.2f}")
print(f"Posterior:  μ = {post_mean:.2f},  σ = {post_std:.2f}  (more certain!)")

### Example: Repeated Bayes updates

What happens when the robot takes **multiple measurements**? Each one is another Bayes
update, the posterior from one step becomes the prior for the next.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
prior_mu = 5.0; prior_sigma = 3.0    # initial (wide) prior
wall_x = 10.0; sensor_sigma = 1.0
measurements = [4.8, 5.1, 4.9, 5.0, 4.7]   # sequence of LiDAR ranges (try adding more)
# ──────────────────────────────────────────────────────────────────────────────

x = np.linspace(0, 10, 500)
belief = stats.norm.pdf(x, prior_mu, prior_sigma)
belief /= np.trapz(belief, x)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, belief, 'steelblue', lw=1.5, ls='--', alpha=0.5, label='initial prior')

colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(measurements)))
for i, z in enumerate(measurements):
    likelihood = stats.norm.pdf(z, wall_x - x, sensor_sigma)
    belief = belief * likelihood
    belief /= np.trapz(belief, x)

    mean_i = np.trapz(x * belief, x)
    std_i = np.sqrt(np.trapz((x - mean_i)**2 * belief, x))
    ax.plot(x, belief, color=colors[i], lw=2,
            label=f'after z{i+1}={z:.1f} (μ={mean_i:.2f}, σ={std_i:.2f})')

ax.set_xlabel("Position (m)"); ax.set_ylabel("Density")
ax.set_title("Sequential Bayes updates — belief gets sharper with each measurement", fontsize=13)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

**Key observations:**
- Bayes rule is the **foundation** of all probabilistic filtering (Kalman, particle, etc.).
- The posterior is **always** between the prior and the likelihood, it fuses both sources of information.
- With more measurements, the posterior becomes **narrower** (more certain).
- Bayes rule works the same way whether the state is 1D position or a high-dimensional SLAM state.

## 5.4 Independence

Two random variables $X$ and $Z$ are **independent** if knowing one tells you nothing about the other:

$$p(x, z) = p(x) \, p(z) \quad \Leftrightarrow \quad p(x \mid z) = p(x)$$

**Conditional independence** is more common in robotics: $X$ and $Y$ are conditionally
independent given $Z$ if:

$$p(x, y \mid z) = p(x \mid z) \, p(y \mid z)$$

This does **not** mean $X$ and $Y$ are independent! It means: *once you know $Z$,
learning $Y$ gives no additional information about $X$*.

### Example: Two LiDAR readings

A robot takes two range measurements $Z_1$ and $Z_2$. Are they independent?
- **Unconditionally**: No! Both depend on the robot's position $X$.
- **Given $X$**: Yes! If we know exactly where the robot is, each reading is just independent sensor noise.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
true_x = 5.0           # true robot position (meters)
sigma_z = 0.8          # sensor noise
n_samples = 1000       # number of paired measurements
sigma_x_prior = 2.0    # prior uncertainty on position
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Case 1: unconditional — x is unknown (sampled from prior)
x_samples = np.random.normal(true_x, sigma_x_prior, n_samples)
z1_unc = 10 - x_samples + np.random.normal(0, sigma_z, n_samples)
z2_unc = 10 - x_samples + np.random.normal(0, sigma_z, n_samples)

# Case 2: conditional on x — position is known
z1_cond = 10 - true_x + np.random.normal(0, sigma_z, n_samples)
z2_cond = 10 - true_x + np.random.normal(0, sigma_z, n_samples)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(z1_unc, z2_unc, s=5, alpha=0.3, c='tomato')
ax.set_xlabel("$z_1$ (m)"); ax.set_ylabel("$z_2$ (m)")
corr_unc = np.corrcoef(z1_unc, z2_unc)[0, 1]
ax.set_title(f"Unconditional: corr = {corr_unc:.3f} (NOT independent)", fontsize=12)
ax.set_aspect('equal')

ax = axes[1]
ax.scatter(z1_cond, z2_cond, s=5, alpha=0.3, c='steelblue')
ax.set_xlabel("$z_1$ (m)"); ax.set_ylabel("$z_2$ (m)")
corr_cond = np.corrcoef(z1_cond, z2_cond)[0, 1]
ax.set_title(f"Conditional on x={true_x}: corr = {corr_cond:.3f} (independent!)", fontsize=12)
ax.set_aspect('equal')

plt.suptitle("$Z_1$ and $Z_2$ are conditionally independent given $X$, but NOT unconditionally",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Key observations:**
- **Conditional independence** is the key assumption behind most SLAM algorithms.
- In Bayes filters: given the current state $x_t$, the measurement $z_t$ is independent of all past measurements.
- This is the **Markov assumption**: $p(z_t \mid x_t, z_{1:t-1}) = p(z_t \mid x_t)$.
- Without this assumption, the computation would grow exponentially with each time step.

## 5.5 Expectation and Variance

The **expectation** (mean) of a random variable is its average value:

$$E[X] = \int x \, p(x) \, dx = \mu$$

The **variance** measures spread:

$$\text{Var}[X] = E[(X - \mu)^2] = E[X^2] - (E[X])^2 = \sigma^2$$

Key properties:
- $E[aX + b] = aE[X] + b$ (linearity)
- $\text{Var}[aX + b] = a^2 \text{Var}[X]$
- For independent $X, Y$: $\text{Var}[X + Y] = \text{Var}[X] + \text{Var}[Y]$

### Example: Estimating position from multiple noisy sensors

A robot has three range sensors with different noise levels. Each gives a noisy estimate
of the same distance. We can combine them, the result has **lower variance** than any individual sensor.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
true_distance = 5.0
sigma1 = 2.0    # sensor 1 noise (cheap sensor)
sigma2 = 1.0    # sensor 2 noise (medium sensor)
sigma3 = 0.5    # sensor 3 noise (expensive sensor)
n_trials = 5000
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
z1 = np.random.normal(true_distance, sigma1, n_trials)
z2 = np.random.normal(true_distance, sigma2, n_trials)
z3 = np.random.normal(true_distance, sigma3, n_trials)

# Simple average
z_avg = (z1 + z2 + z3) / 3

# Optimal weighted average (inverse-variance weighting)
w1, w2, w3 = 1/sigma1**2, 1/sigma2**2, 1/sigma3**2
w_total = w1 + w2 + w3
z_optimal = (w1*z1 + w2*z2 + w3*z3) / w_total
sigma_optimal = np.sqrt(1 / w_total)

fig, ax = plt.subplots(figsize=(12, 5))
x_range = np.linspace(true_distance - 5, true_distance + 5, 300)

for z, s, label, color in [
    (z1, sigma1, f'Sensor 1 (σ={sigma1})', 'lightcoral'),
    (z2, sigma2, f'Sensor 2 (σ={sigma2})', 'lightskyblue'),
    (z3, sigma3, f'Sensor 3 (σ={sigma3})', 'lightgreen')]:
    ax.hist(z, bins=50, density=True, alpha=0.3, color=color, label=label)

ax.hist(z_avg, bins=50, density=True, alpha=0.4, color='orange',
        label=f'Simple avg (σ={np.std(z_avg):.3f})')
ax.hist(z_optimal, bins=50, density=True, alpha=0.4, color='tomato',
        label=f'Optimal weighted (σ={np.std(z_optimal):.3f}, theory={sigma_optimal:.3f})')
ax.axvline(true_distance, color='k', ls='--', lw=2, label=f'true = {true_distance}')
ax.set_xlabel("Distance estimate (m)"); ax.set_ylabel("Density")
ax.set_title("Combining noisy sensors reduces variance", fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Individual sensor std devs: {sigma1:.2f}, {sigma2:.2f}, {sigma3:.2f}")
print(f"Simple average std dev:     {np.std(z_avg):.3f}")
print(f"Optimal weighted std dev:   {np.std(z_optimal):.3f}  (theory: {sigma_optimal:.3f})")
print(f"\nOptimal weights: w1={w1/w_total:.3f}, w2={w2/w_total:.3f}, w3={w3/w_total:.3f}")
print("(The best sensor gets the highest weight — this is what the Kalman filter does!)")

### Example: Law of Total Expectation and Variance

A robot operates in two modes: **indoor** (slow, low noise) and **outdoor** (fast, high noise).
The overall distribution of speed is a *mixture*, not Gaussian even if each mode is.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
p_indoor = 0.6           # probability of indoor mode
mu_indoor = 0.5          # mean speed indoor (m/s)
sigma_indoor = 0.15      # speed noise indoor

mu_outdoor = 2.0         # mean speed outdoor (m/s)
sigma_outdoor = 0.6      # speed noise outdoor
# ──────────────────────────────────────────────────────────────────────────────

p_outdoor = 1 - p_indoor
x = np.linspace(-0.5, 4, 500)

pdf_indoor = stats.norm.pdf(x, mu_indoor, sigma_indoor)
pdf_outdoor = stats.norm.pdf(x, mu_outdoor, sigma_outdoor)
pdf_mixture = p_indoor * pdf_indoor + p_outdoor * pdf_outdoor

# Law of total expectation: E[X] = E[E[X|M]]
E_total = p_indoor * mu_indoor + p_outdoor * mu_outdoor

# Law of total variance: Var[X] = E[Var[X|M]] + Var[E[X|M]]
E_var = p_indoor * sigma_indoor**2 + p_outdoor * sigma_outdoor**2
var_E = p_indoor * (mu_indoor - E_total)**2 + p_outdoor * (mu_outdoor - E_total)**2
V_total = E_var + var_E

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(x, pdf_mixture, alpha=0.3, color='steelblue')
ax.plot(x, pdf_mixture, 'steelblue', lw=2, label='mixture $p(v)$')
ax.plot(x, p_indoor * pdf_indoor, 'tomato', lw=1.5, ls='--', label=f'indoor ({p_indoor:.0%})')
ax.plot(x, p_outdoor * pdf_outdoor, 'orange', lw=1.5, ls='--', label=f'outdoor ({p_outdoor:.0%})')
ax.axvline(E_total, color='k', ls=':', lw=2, label=f'E[v] = {E_total:.2f} m/s')
ax.set_xlabel("Speed (m/s)"); ax.set_ylabel("Density")
ax.set_title("Mixture distribution — not Gaussian!", fontsize=13)
ax.legend(); plt.tight_layout(); plt.show()

print(f"E[V] = {E_total:.3f} m/s")
print(f"Var[V] = E[Var[V|M]] + Var[E[V|M]] = {E_var:.4f} + {var_E:.4f} = {V_total:.4f}")
print(f"Std[V] = {np.sqrt(V_total):.3f} m/s")

**Key observations:**
- **Inverse-variance weighting** is the optimal way to combine independent estimates, this is the core idea behind the Kalman filter's gain.
- The **law of total variance** $\text{Var}[X] = E[\text{Var}[X \mid M]] + \text{Var}[E[X \mid M]]$ decomposes uncertainty into "within-mode" and "between-mode" parts.
- Mixture distributions arise naturally in robotics (multi-modal beliefs, data association ambiguity) and cannot be well-approximated by a single Gaussian.

---

## Exercises

### Exercise 5.1: Normalize a discrete distribution

A robot's position can be one of 5 cells. After a sensor update, the unnormalized belief is
$\tilde{p} = [0.1, 0.5, 0.3, 0.8, 0.2]$. Normalize it so it sums to 1. What is $p(x = 4)$?

In [ ]:
# Your code here
p_tilde = np.array([0.1, 0.5, 0.3, 0.8, 0.2])
# p = p_tilde / sum(p_tilde)

### Exercise 5.2: Bayes rule with discrete states

A robot is equally likely to be in room A or room B.
- In room A, there's a 90% chance of seeing a red landmark.
- In room B, there's a 30% chance of seeing a red landmark.

The robot sees a red landmark. What is the posterior probability it's in room A?
Compute using Bayes rule.

In [ ]:
# Your code here
p_A = 0.5; p_B = 0.5
p_red_given_A = 0.9; p_red_given_B = 0.3
# p(A | red) = p(red | A) * p(A) / p(red)
# p(red) = p(red|A)*p(A) + p(red|B)*p(B)

### Exercise 5.3: Sensor fusion, optimal weights

Three range sensors measure the same distance:
- Sensor A: $z_A = 4.8$ m, $\sigma_A = 1.0$ m
- Sensor B: $z_B = 5.3$ m, $\sigma_B = 0.5$ m
- Sensor C: $z_C = 5.1$ m, $\sigma_C = 2.0$ m

Compute the inverse-variance weighted estimate and its uncertainty.

In [ ]:
# Your code here
z = np.array([4.8, 5.3, 5.1])
sigma = np.array([1.0, 0.5, 2.0])
# weights = 1/sigma**2
# z_fused = sum(weights * z) / sum(weights)
# sigma_fused = sqrt(1 / sum(weights))

### Exercise 5.4: Sequential Bayes updates

A robot's prior is $p(x) = \mathcal{N}(0, 10^2)$ (very uncertain).
It receives 10 GPS measurements, each drawn from $\mathcal{N}(x_{true}, 2^2)$ with $x_{true} = 3.0$.

Simulate the 10 measurements and apply Bayes rule sequentially.
Plot the belief after each update. How many measurements does it take for $\sigma < 1$?

In [ ]:
# Your code here
np.random.seed(0)
x_true = 3.0
measurements = np.random.normal(x_true, 2.0, 10)
# Start with prior N(0, 100)
# For Gaussian prior + Gaussian likelihood, the posterior is also Gaussian:
# mu_post = (mu_prior/sigma_prior^2 + z/sigma_z^2) / (1/sigma_prior^2 + 1/sigma_z^2)
# sigma_post^2 = 1 / (1/sigma_prior^2 + 1/sigma_z^2)

### Exercise 5.5: Conditional independence test (challenge)

Generate $N = 2000$ samples from the following model:
- $X \sim \mathcal{N}(5, 2^2)$
- $Z_1 = X + \epsilon_1$, where $\epsilon_1 \sim \mathcal{N}(0, 1)$
- $Z_2 = X + \epsilon_2$, where $\epsilon_2 \sim \mathcal{N}(0, 1)$

1. Compute `np.corrcoef(z1, z2)`, are $Z_1$ and $Z_2$ correlated?
2. Now condition on $X$: for samples where $4.9 < X < 5.1$, compute the correlation of $Z_1$ and $Z_2$.
3. Explain why the conditional correlation is near zero.

In [ ]:
# Your code here
np.random.seed(42)
N = 2000
# X = np.random.normal(5, 2, N)
# Z1 = X + np.random.normal(0, 1, N)
# Z2 = X + np.random.normal(0, 1, N)